# Command Primitives Brain Playground
Same training framework as other playgrounds; this one switches control mode.


In [1]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_headless.py").exists() and (REPO_ROOT.parent / "train_headless.py").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from brains.config import load_runtime_spec
from brains.jax_trainer import apply_runtime_spec
from brains.models import (
    NotebookModel,
    apply_notebook_model,
    register_notebook_model,
)
from brains.models.notebook_framework import train_and_save_with_progress



In [2]:
MODEL_TYPE = "notebook_command_brain_v1"
LOG_ID = datetime.now(timezone.utc).strftime("command_brain_%Y%m%dT%H%M%SZ")
TRAIN_GENERATIONS = 2
SEED = 11
COMMANDS = ("stand", "trot", "turn_left", "turn_right", "back_up")

base_spec = load_runtime_spec(REPO_ROOT / "configs/default.yaml")
model = NotebookModel(
    model_type=MODEL_TYPE,
    architecture="linear_commands",
    description="Command-level policy (command logits + hold-duration head).",
    input_size=48,
    output_size=len(COMMANDS) + 1,
    parameter_count=None,
    policy_entrypoint="brains.models.reference_linear_plugin:build_linear_policy",
    control_mode="command_primitives",
    command_vocabulary=COMMANDS,
    command_update_interval_s=0.10,
    command_default_duration_s=0.25,
    command_max_duration_s=0.80,
)
register_notebook_model(model, spec=base_spec, persist=True)

spec = apply_notebook_model(base_spec, model)
spec = replace(spec, name="command-brain-default")
apply_runtime_spec(spec)

{"model": spec.model.type, "architecture": spec.model.architecture, "control_mode": spec.control.mode, "commands": spec.control.command_vocabulary}



W0501 11:10:40.101303 2651673 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


{'model': 'notebook_command_brain_v1',
 'architecture': 'linear_commands',
 'control_mode': 'command_primitives',
 'commands': ('stand', 'trot', 'turn_left', 'turn_right', 'back_up')}

In [3]:
result = train_and_save_with_progress(
    spec=spec,
    repo_root=REPO_ROOT,
    model_type=MODEL_TYPE,
    log_id=LOG_ID,
    seed=SEED,
    generations=TRAIN_GENERATIONS,
    print_step_progress=False,
)
result



generation 1/2 start


generation 1/2 done | mean_reward=5.4512 | best_reward=10.0357


generation 2/2 start


generation 2/2 done | mean_reward=14.0261 | best_reward=10.0357


{'model_id': 'notebook_command_brain_v1_command_brain_20260501T151039Z',
 'log_id': 'command_brain_20260501T151039Z',
 'latest': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/checkpoints/notebook_command_brain_v1_command_brain_20260501T151039Z/latest.npz',
 'generation': 2,
 'mean_reward': 14.026116371154785,
 'best_reward': 10.03565788269043,
 'elapsed_s': 97.24691137501213,
 'visible_artifacts': ['notebook_command_brain_v1_command_brain_20260501T151039Z',
  'notebook_shared_trunk_v1_shared_trunk_20260501T130232Z']}